# Predicted Epitopes of *Trypanosoma cruzi* based on Phage Display Data Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing entities by their `@id` fields according to the Croissant schema.

### Dataset Source
Schema: [https://sen.science/doi/10.71728/senscience.etbd-dths/fair2.json](https://sen.science/doi/10.71728/senscience.etbd-dths/fair2.json)


In [ ]:
# Ensure mlcroissant and visualization libraries are installed
!pip install mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.etbd-dths/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Metadata is exposed as an object. Access fields directly.
metadata = dataset.metadata.to_json()

print(f"Dataset name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Published: {metadata['datePublished']}")
print(f"Identifier: {metadata['identifier']}")
print(f"License: {metadata['license']}")
print(f"Keywords: {metadata['keywords']}")

## 2. Data Overview
Review available record sets, fields, and their IDs referenced by `@id` fields.

**Note:** In Croissant, each entity (record set, field, column, etc.) has a unique `@id`.

In [ ]:
# List available record sets by their @id
recordsets = dataset.metadata.recordSet
if isinstance(recordsets, list):
    print(f"Record sets (@id): {[rs['@id'] if isinstance(rs, dict) else rs for rs in recordsets]}")
else:
    print(f"Record set (@id): {recordsets['@id'] if isinstance(recordsets, dict) else recordsets}")

# If dataset has no recordSet directly, attempt to enumerate known record sets
def list_record_sets():
    rs_list = []
    if hasattr(dataset.metadata, 'recordSet'):
        rs = dataset.metadata.recordSet
        if isinstance(rs, list):
            for r in rs:
                rs_list.append(r['@id'] if isinstance(r, dict) and '@id' in r else r)
        elif isinstance(rs, dict):
            rs_list.append(rs['@id'])
        elif isinstance(rs, str):
            rs_list.append(rs)
    return rs_list

record_set_ids = list_record_sets()
if record_set_ids:
    print("Available record set @ids:")
    for rsid in record_set_ids:
        print(f"  - {rsid}")
else:
    print("No record sets exposed directly in metadata. Attempting dynamic inspection...")
    # Try to enumerate from dataset.records()
    all_count = 0
    try:
        # Use mlcroissant API to list record sets if available
        print("Sample records by recordSet @id (if any):")
        # This demo will print up to 2 example records per record_set
        for rsid in dataset.record_sets:
            print(f"\nRecord set: {rsid}")
            records = list(dataset.records(record_set=rsid))
            for rec in records[:2]:
                print(rec)
            all_count += 1
        if all_count == 0:
            print("No explicit record sets found.")
    except Exception as e:
        print("Unable to enumerate record sets dynamically: ", e)

## 3. Data Extraction
Load records from each record set into DataFrames using their `@id` fields.

All record sets, fields, and columns are referenced by their `@id`.

In [ ]:
# Find all recordSet @ids
if hasattr(dataset, 'record_sets'):
    record_set_ids = list(dataset.record_sets)
else:
    record_set_ids = list_record_sets()

# Extract records from each record set
dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records for RecordSet @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns (@id): {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

# Choose a record set for further analysis.
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nSelected record set for analysis: {main_record_set_id}")
else:
    print("No dataframes available for analysis.")

## 4. Exploratory Data Analysis (EDA)
Process numeric fields, filter records, normalize, and group data using entity `@id` references.

**Note:** Replace field `@id`s in code as appropriate for your chosen record set. If the record set contains epitope predictions, antigenic regions, or peptide statistics, use their columns for analysis.

In [ ]:
# Begin EDA on selected record set
df = dataframes.get(main_record_set_id, pd.DataFrame())
if df.empty:
    print("No records loaded for EDA.")
else:
    # Print available columns
    print("Available columns (@id):")
    for col in df.columns:
        print(f" - {col}")

    # Select a numeric field for analysis
    # For demonstration, auto-select first numeric column
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        print(f"\nSelected numeric field (@id): {numeric_field_id}")
        
        threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field
        group_field_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id]
        if group_field_candidates:
            group_field_id = group_field_candidates[0]
            print(f"\nGrouping records by {group_field_id} (@id):")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric fields found in the selected record set.")

## 5. Visualization
Visualize distributions and relationships between numeric and group fields, referencing `@id` for all entities.

In [ ]:
# Plot numeric field distribution and group comparison
if not df.empty and numeric_cols:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Grouped bar plot if a group field is present
    if group_field_candidates:
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()


## 6. Conclusion
- Explored the FAIR² dataset using `mlcroissant`, referencing all entities by their `@id` fields.
- Loaded metadata and records from Croissant schema.
- Performed EDA with filtering, normalization, grouping, and visualizations.
- The dataset enables downstream immunological and genomic studies of Chagas disease, supporting analysis of predicted epitopes and antigenic regions.

For further analysis, consult the Croissant schema for additional metadata and field definitions, and reference all entities using their unique `@id`.
